In [4]:
pip install pandas "numpy<2.4" matplotlib seaborn scikit-learn ydata-profiling "setuptools<81" ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
print(sys.executable)

from ydata_profiling import ProfileReport

/Users/lipikam/Desktop/MinorProject/.venv-1/bin/python


/var/folders/wm/1jg9q6hn6qb190_w715mhp100000gn/T/ipykernel_11756/706283429.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [5]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
project_root = Path("/Users/lipikam/Desktop/MinorProject/ANSC4040MiniProject")
file_path = project_root / "Data_set_prep_assignment_1.csv"

# Keep AnimalId as text so large identifiers are not rounded by floating-point conversion.
df = pd.read_csv(file_path, low_memory=False, dtype={"AnimalId": "string"})
df = df.rename(columns={
    "Avgmilkflow": "AverageMilkFlowKgPerMin",
    "Flow30_60Session": "MilkFlow30To60SecondsKgPerMin",
    "YieldFirst2Min_Session": "MilkYieldFirst2MinutesKg",
    "YieldSession": "TotalMilkYieldSessionKg",
    "DurationSession_sec": "MilkingDurationSessionSeconds",
    "milking": "MilkingSession"
})
df["EventDate"] = pd.to_datetime(df["EventDate"], errors="coerce")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 8,495,421
Columns: 11


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds,MilkingSession
0,-8839528597343470980,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5365332022005618386,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,<NA>,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7750423760892252046,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,<NA>,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


In [ ]:
profile_report = ProfileReport(
    df,
    title="Data Profile Report",
    explorative=True
)

profile_report.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:14<00:00,  1.27s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
import os
import sys

print("Kernel PID:", os.getpid())
print("Python executable:", sys.executable)

Kernel PID: 12155
Python executable: /Users/lipikam/Desktop/MinorProject/.venv-1/bin/python


In [2]:
print("df exists:", "df" in globals())
print("ProfileReport exists:", "ProfileReport" in globals())

df exists: False
ProfileReport exists: False


In [2]:
profile_report.to_file(
    "/Users/lipikam/Desktop/MinorProject/ANSC4040MiniProject/data_profile_report.html"
)

NameError: name 'profile_report' is not defined

Data Preprocessing

In [7]:
# Review data types and potentially invalid values before applying preprocessing rules 
# and understand the missingness of the data + Summary of the data.

numeric_columns = [
    "LactationNumber",
    "DaysInMilk",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg",
    "TotalMilkYieldSessionKg",
    "MilkingDurationSessionSeconds"
]

data_quality = pd.DataFrame({
    "DataType": df.dtypes.astype(str),
    "MissingCount": df.isna().sum(),
    "MissingPercent": (df.isna().mean() * 100).round(2),
    "UniqueValues": df.nunique(dropna=True),
    "NegativeValueCount": (
        df[numeric_columns].lt(0).sum()
        .reindex(df.columns, fill_value=0)
    ),
    "ZeroValueCount": (
        df[numeric_columns].eq(0).sum()
        .reindex(df.columns, fill_value=0)
    )
})

data_quality["InvalidDateCount"] = 0
data_quality.loc["EventDate", "InvalidDateCount"] = df["EventDate"].isna().sum()

data_quality["OutsideExpectedSessionCount"] = 0
data_quality.loc["MilkingSession", "OutsideExpectedSessionCount"] = (
    ~df["MilkingSession"].isin([1, 2, 3])
).sum()

display(data_quality.sort_values("MissingPercent", ascending=False))

,DataType,MissingCount,MissingPercent,UniqueValues,NegativeValueCount,ZeroValueCount,InvalidDateCount,OutsideExpectedSessionCount
AnimalId,string,1701003,20.02,9087,0,0,0,0
LactationNumber,float64,1701003,20.02,12,0,0,0,0
DaysInMilk,float64,1701007,20.02,870,0,0,0,0
ReproductionStatus,object,1701003,20.02,4,0,0,0,0
EventDate,datetime64[ns],0,0.00,830,0,0,0,0
AverageMilkFlowKgPerMin,float64,172,0.00,219,0,215,0,0
MilkFlow30To60SecondsKgPerMin,float64,0,0.00,102,0,3155,0,0
MilkYieldFirst2MinutesKg,float64,0,0.00,1675,0,0,0,0
TotalMilkYieldSessionKg,float64,0,0.00,1010,0,0,0,0
MilkingDurationSessionSeconds,int64,0,0.00,604,0,0,0,0


In [8]:
#more summarisation of the dataset
numeric_columns = df.select_dtypes(include="number").columns

dataset_summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total columns",
        "Minimum event date",
        "Maximum event date",
        "Number of unique AnimalId values"
    ],
    "Value": [
        len(df),
        df.shape[1],
        df["EventDate"].min(),
        df["EventDate"].max(),
        df["AnimalId"].nunique(dropna=True)
    ]
})
display(dataset_summary)

,Metric,Value
0,Total records,8495421
1,Total columns,11
2,Minimum event date,2019-06-14 00:00:00
3,Maximum event date,2021-10-24 00:00:00
4,Number of unique AnimalId values,9087


In [11]:
# Preserve the original dataset and create a temporary modeling copy

df_master = df.copy()

flow_column = "AverageMilkFlowKgPerMin"

df_model = df_master.dropna(subset=[flow_column]).copy()

print(f"Master rows: {len(df_master):,}")
print(f"Modeling rows: {len(df_model):,}")
print(f"Rows excluded from modeling : {len(df_master) - len(df_model):,}")

Master rows: 8,495,421
Modeling rows: 8,495,249
Rows excluded from modeling : 172


In [12]:
# Remove exact duplicates from the existing modeling dataset

rows_before = len(df_model)

duplicate_rows_beyond_first = int(
    df_model.duplicated(keep="first").sum()
)

duplicate_groups = int(
    df_model.groupby(
        list(df_model.columns),
        dropna=False,
        sort=False
    ).size().gt(1).sum()
)

df_model = df_model.drop_duplicates(keep="first").copy()

print(f"Rows before removing duplicates: {rows_before:,}")
print(f"Duplicate groups: {duplicate_groups:,}")
print(f"Duplicate rows removed: {duplicate_rows_beyond_first:,}")
print(f"Rows after removing duplicates: {len(df_model):,}")

Rows before removing duplicates: 8,495,249
Duplicate groups: 50,834
Duplicate rows removed: 60,653
Rows after removing duplicates: 8,434,596


In [13]:
# Detect potential outliers using the IQR rule

outlier_columns = [
    "LactationNumber",
    "DaysInMilk",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg",
    "TotalMilkYieldSessionKg",
    "MilkingDurationSessionSeconds"
]

outlier_summary = []

for column in outlier_columns:
    q1 = df_model[column].quantile(0.25)
    q3 = df_model[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (df_model[column] < lower_bound)
        | (df_model[column] > upper_bound)
    ).sum()

    outlier_summary.append({
        "Column": column,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "PotentialOutlierCount": outlier_count,
        "PotentialOutlierPercent": round(
            outlier_count / len(df_model) * 100, 4
        )
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(outlier_summary)

,Column,LowerBound,UpperBound,PotentialOutlierCount,PotentialOutlierPercent
0,LactationNumber,-2.000000,6.000000,51936,0.6157
1,DaysInMilk,-165.500000,478.500000,44682,0.5297
2,AverageMilkFlowKgPerMin,1.496855,5.488468,6835,0.0810
3,MilkFlow30To60SecondsKgPerMin,-0.351534,8.048997,1543,0.0183
4,MilkYieldFirst2MinutesKg,1.560358,13.462622,0,0.0000
5,TotalMilkYieldSessionKg,4.082331,23.677522,43332,0.5137
6,MilkingDurationSessionSeconds,84.500000,384.500000,50267,0.5960


In [14]:
numeric_summary = df_model[outlier_columns].agg(
    ["count", "min", "max", "mean", "median", "std"]
).T

display(numeric_summary)

,count,min,max,mean,median,std
LactationNumber,6737767.0,1.000000,12.000000,2.293943,2.000000,1.387633
DaysInMilk,6737763.0,1.000000,870.000000,162.937711,154.000000,105.515783
AverageMilkFlowKgPerMin,8434596.0,0.000000,23.496085,3.525932,3.492661,0.729215
MilkFlow30To60SecondsKgPerMin,8434596.0,0.000000,10.500663,3.751315,3.900894,1.485700
MilkYieldFirst2MinutesKg,8434596.0,2.000342,11.997518,7.491103,7.525097,2.014110
TotalMilkYieldSessionKg,8434596.0,5.034875,54.839318,13.927371,13.743849,3.644127
MilkingDurationSessionSeconds,8434596.0,100.000000,785.000000,236.854545,231.000000,54.837200


In [15]:
# Investigate zero AverageMilkFlowKgPerMin records

zero_flow_rows = df_model[
    df_model["AverageMilkFlowKgPerMin"] == 0
].copy()

zero_flow_summary = pd.DataFrame({
    "ZeroFlowRecords": [len(zero_flow_rows)],
    "KnownIdRecords": [
        zero_flow_rows["AnimalId"].notna().sum()
    ],
    "MissingIdRecords": [
        zero_flow_rows["AnimalId"].isna().sum()
    ],
    "DistinctDates": [
        zero_flow_rows["EventDate"].nunique()
    ],
    "DistinctSessions": [
        zero_flow_rows["MilkingSession"].nunique()
    ]
})

display(zero_flow_summary)

display(
    zero_flow_rows[
        [
            "AnimalId",
            "EventDate",
            "MilkingSession",
            "MilkFlow30To60SecondsKgPerMin",
            "MilkYieldFirst2MinutesKg",
            "TotalMilkYieldSessionKg",
            "MilkingDurationSessionSeconds"
        ]
    ].head(20)
)

,ZeroFlowRecords,KnownIdRecords,MissingIdRecords,DistinctDates,DistinctSessions
0,204,164,40,136,3


,AnimalId,EventDate,MilkingSession,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds
48952,-8950289851158579840,2019-12-18,2,3.801104,6.799350,20.774531,243
123340,6185946476489130314,2020-09-04,1,3.197826,6.023707,15.149985,289
162997,-5790227351559561756,2021-08-31,3,2.698875,5.075699,9.480081,377
219939,618884451670809093,2020-12-17,2,4.300056,6.023707,6.803886,164
226475,-6952558986316616714,2019-12-22,1,4.998588,9.298644,36.650263,253
248542,-7348751580097396124,2019-11-20,2,1.202020,3.701314,7.665711,171
314264,-5056076151021360723,2020-01-16,2,1.601181,3.175147,16.147888,326
328635,-6952558986316616714,2019-12-17,1,3.601523,8.600111,35.062690,227
350637,-7546303925728263247,2020-08-07,3,1.102229,5.375070,8.255381,174
401078,3079249593767300237,2021-07-25,3,5.801446,10.772819,16.420044,202


In [16]:
# Inspect high-end values before deciding whether they are invalid

high_value_checks = {
    "AverageMilkFlowKgPerMin": df_model["AverageMilkFlowKgPerMin"].quantile(0.999),
    "DaysInMilk": df_model["DaysInMilk"].quantile(0.999),
    "MilkingDurationSessionSeconds": df_model[
        "MilkingDurationSessionSeconds"
    ].quantile(0.999)
}

for column, threshold in high_value_checks.items():
    print(f"\n{column}")
    print(f"99.9th-percentile threshold: {threshold:.3f}")
    
    display(
        df_model[df_model[column] >= threshold][
            [
                "AnimalId",
                "EventDate",
                "MilkingSession",
                "LactationNumber",
                "DaysInMilk",
                "AverageMilkFlowKgPerMin",
                "TotalMilkYieldSessionKg",
                "MilkingDurationSessionSeconds"
            ]
        ]
        .sort_values(column, ascending=False)
        .head(20)
    )


AverageMilkFlowKgPerMin
99.9th-percentile threshold: 5.398


,AnimalId,EventDate,MilkingSession,LactationNumber,DaysInMilk,AverageMilkFlowKgPerMin,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds
2315446,7502706608343226189,2020-03-13,2,2.0,187.0,23.496085,41.503702,182
7145852,<NA>,2020-03-13,2,NaN,NaN,23.496085,47.173606,215
6694148,4264723567066082770,2020-03-13,2,2.0,296.0,23.496085,43.227353,196
966151,-852563118874554451,2020-03-13,2,3.0,162.0,23.405366,39.870769,207
4093835,-2698874661650367761,2020-03-13,2,2.0,48.0,23.405366,47.491121,260
527419,<NA>,2020-03-22,2,NaN,NaN,23.405366,40.505799,164
1876443,5307502696978313350,2020-03-12,2,2.0,83.0,23.405366,51.392016,242
5561738,1737820796196306887,2020-03-13,2,2.0,203.0,23.405366,48.307587,251
1279392,<NA>,2020-03-13,2,NaN,NaN,23.405366,44.588130,257
7350352,<NA>,2020-03-22,2,NaN,NaN,23.405366,43.318071,183



DaysInMilk
99.9th-percentile threshold: 631.000


,AnimalId,EventDate,MilkingSession,LactationNumber,DaysInMilk,AverageMilkFlowKgPerMin,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds
5050371,6873203251618857930,2021-04-04,1,1.0,870.0,3.220506,6.985322,128
5606196,6873203251618857930,2021-04-04,2,1.0,870.0,2.902991,6.894604,139
1256081,6873203251618857930,2021-04-03,2,1.0,869.0,2.902991,6.531730,134
2938663,6873203251618857930,2021-04-03,3,1.0,869.0,2.812273,6.985322,147
5853316,6873203251618857930,2021-04-02,1,1.0,868.0,2.222603,7.665711,204
6583198,6873203251618857930,2021-04-02,3,1.0,868.0,2.585477,6.622449,152
1291044,6873203251618857930,2021-04-01,1,1.0,867.0,3.311224,7.892507,140
1275055,6873203251618857930,2021-04-01,2,1.0,867.0,2.721554,6.667808,145
1041581,6873203251618857930,2021-03-31,1,1.0,866.0,2.585477,6.350293,142
8340447,6873203251618857930,2021-03-31,2,1.0,866.0,2.812273,6.531730,136



MilkingDurationSessionSeconds
99.9th-percentile threshold: 480.000


,AnimalId,EventDate,MilkingSession,LactationNumber,DaysInMilk,AverageMilkFlowKgPerMin,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds
2333804,-4087174035999327230,2020-09-12,2,4.0,6.0,1.406136,18.325132,785
3647464,-550670112082332253,2020-06-16,2,4.0,113.0,2.993710,40.233643,782
2923168,-2788247857510246461,2021-01-10,3,3.0,227.0,2.086525,27.442338,781
3176214,8953141912054890343,2020-09-25,2,3.0,210.0,2.494758,33.339039,775
4546101,<NA>,2021-01-01,3,NaN,NaN,2.721554,35.380205,773
2892855,9117999581887597177,2019-06-20,3,5.0,79.0,2.313321,29.574223,765
3674840,<NA>,2020-03-29,3,NaN,NaN,1.179340,15.694296,764
4775039,<NA>,2020-09-25,2,NaN,NaN,2.812273,36.105953,764
2272625,-514841208892209311,2020-06-15,1,1.0,254.0,1.814369,23.677522,763
1529999,2489612738664029105,2020-06-18,2,1.0,46.0,1.905088,15.875733,761


In [17]:
animal_id = "6873203251618857930"

display(
    df_model[df_model["AnimalId"] == animal_id][
        [
            "AnimalId",
            "EventDate",
            "MilkingSession",
            "LactationNumber",
            "DaysInMilk",
            "AverageMilkFlowKgPerMin",
            "TotalMilkYieldSessionKg",
            "MilkingDurationSessionSeconds"
        ]
    ]
    .sort_values(["EventDate", "MilkingSession"])
)

,AnimalId,EventDate,MilkingSession,LactationNumber,DaysInMilk,AverageMilkFlowKgPerMin,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds
7245647,6873203251618857930,2019-06-15,2,1.0,211.0,4.490564,16.873636,222
4752650,6873203251618857930,2019-06-15,3,1.0,211.0,4.218409,14.560315,205
1091304,6873203251618857930,2019-06-16,2,1.0,212.0,4.309128,16.556122,231
5018757,6873203251618857930,2019-06-17,1,1.0,213.0,4.218409,15.467500,217
2421722,6873203251618857930,2019-06-17,2,1.0,213.0,4.082331,15.013907,219
...,...,...,...,...,...,...,...,...
6583198,6873203251618857930,2021-04-02,3,1.0,868.0,2.585477,6.622449,152
1256081,6873203251618857930,2021-04-03,2,1.0,869.0,2.902991,6.531730,134
2938663,6873203251618857930,2021-04-03,3,1.0,869.0,2.812273,6.985322,147
5050371,6873203251618857930,2021-04-04,1,1.0,870.0,3.220506,6.985322,128


In [18]:
high_dim_summary = (
    df_model[df_model["DaysInMilk"] > 600]
    .groupby("AnimalId", dropna=False)
    .agg(
        RecordCount=("DaysInMilk", "size"),
        MinimumDaysInMilk=("DaysInMilk", "min"),
        MaximumDaysInMilk=("DaysInMilk", "max"),
        LactationNumbers=("LactationNumber", "nunique")
    )
    .sort_values("MaximumDaysInMilk", ascending=False)
)

display(high_dim_summary.head(20))

print(f"Records with DaysInMilk > 600: {(df_model['DaysInMilk'] > 600).sum():,}")
print(f"Animals with DaysInMilk > 600: {high_dim_summary.shape[0]:,}")

,RecordCount,MinimumDaysInMilk,MaximumDaysInMilk,LactationNumbers
AnimalId,,,,
6873203251618857930,542,601.0,870.0,1
-8955758822887219123,570,603.0,863.0,1
-4073466570224705704,390,601.0,862.0,1
3973755457434747114,368,615.0,808.0,1
-4837861458145053899,362,601.0,806.0,1
-4557964833378043555,452,601.0,798.0,1
-1972272306200278355,423,601.0,784.0,1
-2334618899818048928,217,642.0,763.0,1
3533441765383033645,345,601.0,749.0,1


Records with DaysInMilk > 600: 9,978
Animals with DaysInMilk > 600: 72


In [19]:
numeric_columns = [
    "DaysInMilk",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg",
    "TotalMilkYieldSessionKg",
    "MilkingDurationSessionSeconds"
]

for column in numeric_columns:
    values = df_model[column].dropna()
    mean = values.mean()
    std = values.std()

    z_scores = (df_model[column] - mean) / std
    flagged = df_model[z_scores.abs() > 3].copy()

    print(f"\n{column}")
    print(f"Flagged records: {len(flagged):,}")

    display(
        flagged[
            [
                "AnimalId",
                "EventDate",
                "MilkingSession",
                column
            ]
        ]
        .sort_values(column, ascending=False)
        .head(20)
    )


DaysInMilk
Flagged records: 44,123


,AnimalId,EventDate,MilkingSession,DaysInMilk
5606196,6873203251618857930,2021-04-04,2,870.0
5050371,6873203251618857930,2021-04-04,1,870.0
1256081,6873203251618857930,2021-04-03,2,869.0
2938663,6873203251618857930,2021-04-03,3,869.0
5853316,6873203251618857930,2021-04-02,1,868.0
6583198,6873203251618857930,2021-04-02,3,868.0
1275055,6873203251618857930,2021-04-01,2,867.0
1291044,6873203251618857930,2021-04-01,1,867.0
8340447,6873203251618857930,2021-03-31,2,866.0
1041581,6873203251618857930,2021-03-31,1,866.0



AverageMilkFlowKgPerMin
Flagged records: 3,803


,AnimalId,EventDate,MilkingSession,AverageMilkFlowKgPerMin
2315446,7502706608343226189,2020-03-13,2,23.496085
7145852,<NA>,2020-03-13,2,23.496085
6694148,4264723567066082770,2020-03-13,2,23.496085
6546382,<NA>,2020-03-13,2,23.405366
1279392,<NA>,2020-03-13,2,23.405366
527419,<NA>,2020-03-22,2,23.405366
4093835,-2698874661650367761,2020-03-13,2,23.405366
7350352,<NA>,2020-03-22,2,23.405366
5607118,<NA>,2020-03-22,2,23.405366
1664214,<NA>,2020-03-13,2,23.405366



MilkFlow30To60SecondsKgPerMin
Flagged records: 839


...

,AnimalId,EventDate,MilkingSession,MilkYieldFirst2MinutesKg



TotalMilkYieldSessionKg
Flagged records: 20,642


,AnimalId,EventDate,MilkingSession,TotalMilkYieldSessionKg
6475041,-4534294872927679798,2020-03-14,2,54.839318
831647,-2360489827573713067,2020-03-13,2,54.839318
3638567,5333208963937859999,2020-03-16,1,54.703240
5808299,4322748837188646076,2021-09-18,2,54.521803
4564751,-5237984363523256449,2020-03-15,3,54.431084
3977837,6805122606403239110,2020-03-16,1,54.385725
6933478,-8096126032068098318,2020-10-20,2,54.295007
8210341,8951233476601031265,2020-03-15,3,54.249647
4381719,-9094170526775755474,2020-03-19,2,54.158929
4634753,-1703950033004114534,2020-03-19,2,54.158929



MilkingDurationSessionSeconds
Flagged records: 31,962


,AnimalId,EventDate,MilkingSession,MilkingDurationSessionSeconds
2333804,-4087174035999327230,2020-09-12,2,785
3647464,-550670112082332253,2020-06-16,2,782
2923168,-2788247857510246461,2021-01-10,3,781
3176214,8953141912054890343,2020-09-25,2,775
4546101,<NA>,2021-01-01,3,773
2892855,9117999581887597177,2019-06-20,3,765
4775039,<NA>,2020-09-25,2,764
3674840,<NA>,2020-03-29,3,764
2272625,-514841208892209311,2020-06-15,1,763
1529999,2489612738664029105,2020-06-18,2,761


In [20]:
df_model["PotentialOutlier"] = False

for column in numeric_columns:
    mean = df_model[column].mean()
    std = df_model[column].std()

    z_score = (df_model[column] - mean) / std
    df_model.loc[z_score.abs() > 3, "PotentialOutlier"] = True

print(
    f"Rows flagged in at least one numeric column: "
    f"{df_model['PotentialOutlier'].sum():,}"
)

Rows flagged in at least one numeric column: 97,998


In [21]:
df_model["ExpectedMilkYieldKg"] = (
    df_model["AverageMilkFlowKgPerMin"]
    * df_model["MilkingDurationSessionSeconds"]
    / 60
)

df_model["YieldDifferenceKg"] = (
    df_model["TotalMilkYieldSessionKg"]
    - df_model["ExpectedMilkYieldKg"]
)

display(
    df_model[
        [
            "AverageMilkFlowKgPerMin",
            "MilkingDurationSessionSeconds",
            "TotalMilkYieldSessionKg",
            "ExpectedMilkYieldKg",
            "YieldDifferenceKg"
        ]
    ]
    .describe()
)

,AverageMilkFlowKgPerMin,MilkingDurationSessionSeconds,TotalMilkYieldSessionKg,ExpectedMilkYieldKg,YieldDifferenceKg
count,8.434596e+06,8.434596e+06,8.434596e+06,8.434596e+06,8.434596e+06
mean,3.525932e+00,2.368545e+02,1.392737e+01,1.375621e+01,1.711635e-01
std,7.292146e-01,5.483720e+01,3.644127e+00,3.681032e+00,9.582646e-01
min,0.000000e+00,1.000000e+02,5.034875e+00,0.000000e+00,-1.094450e+02
25%,2.993710e+00,1.970000e+02,1.143053e+01,1.123473e+01,8.542656e-02
50%,3.492661e+00,2.310000e+02,1.374385e+01,1.357148e+01,1.799250e-01
75%,3.991613e+00,2.720000e+02,1.632933e+01,1.611765e+01,2.797153e-01
max,2.349608e+01,7.850000e+02,5.483932e+01,1.459592e+02,3.878215e+01


In [22]:
fingerprint_columns = [
    "EventDate",
    "AverageMilkFlowKgPerMin",
    "MilkFlow30To60SecondsKgPerMin",
    "MilkYieldFirst2MinutesKg",
    "TotalMilkYieldSessionKg",
    "MilkingDurationSessionSeconds",
    "MilkingSession"
]

known_rows = df_model[df_model["AnimalId"].notna()].copy()

known_rows["Fingerprint"] = (
    known_rows[fingerprint_columns]
    .astype("string")
    .fillna("<MISSING>")
    .agg("|".join, axis=1)
)

fingerprint_id_counts = (
    known_rows.groupby("Fingerprint")["AnimalId"]
    .nunique()
)

unique_fingerprints = fingerprint_id_counts[
    fingerprint_id_counts == 1
].index

validation_rows = known_rows[
    known_rows["Fingerprint"].isin(unique_fingerprints)
].copy()

fingerprint_to_id = (
    validation_rows
    .drop_duplicates("Fingerprint")
    .set_index("Fingerprint")["AnimalId"]
)

validation_rows["PredictedAnimalId"] = (
    validation_rows["Fingerprint"].map(fingerprint_to_id)
)

validation_rows["Correct"] = (
    validation_rows["AnimalId"]
    == validation_rows["PredictedAnimalId"]
)

print(f"Validation rows: {len(validation_rows):,}")
print(f"Correct matches: {validation_rows['Correct'].sum():,}")
print(
    f"Exact-match accuracy: "
    f"{validation_rows['Correct'].mean() * 100:.2f}%"
)

Validation rows: 6,737,167
Correct matches: 6,737,167
Exact-match accuracy: 100.00%
